# 03 · ESM-C: frozen-representation PPI benchmark

**Author:** Ryan Dai  
**Task:** binary protein–protein interaction classification using frozen protein representations.

This notebook encodes protein sequences with **ESM-C 600M** by default, constructs symmetric pair features, and trains a multilayer perceptron. It reuses the saved SNOOPPI split CSVs used by the other notebooks; it does not download the dataset or regenerate partitions. `esmc_300m` is an alternative experiment with separate model-specific artifacts.

**Run order:** use a fresh Google Colab GPU runtime and execute sections 1–11 in order. Embedding-cache recovery and classifier checkpoint recovery are built into the preserved implementation.

| Item | Contract |
|---|---|
| Required inputs | `/content/drive/MyDrive/SNOOPPI/splits/{train,validation,test}.csv` |
| Protein columns | `partner_A_sequence`, `partner_B_sequence` |
| Label source | Existing binary `label`, or positive/negative values in `SNOOPPI_final_label` |
| Model selection | Highest validation macro-average precision (macro-AP) |
| Classification threshold | Highest validation macro-F1 on a fixed grid |
| Evaluation | Train, validation, and designated test split; split overlap must be interpreted explicitly |
| Outputs | ESM-C embedding cache, best/last classifier checkpoints, metrics/prediction CSVs, PNG figures, and a run manifest |

**Provenance and validation:** this repository edition preserves every supplied executable cell and its order. Documentation and notebook execution metadata were cleaned; no experiment was rerun and no results are prefilled. Full execution requires the original split files, model access, and a compatible runtime. ProtT5/ESM-2 vectors and classifier weights cannot substitute for ESM-C artifacts.


## 1. Environment, random seed, and experiment paths

The setup cell pins `esm==3.2.1` and `transformers==4.48.1`, while constraining several scientific packages to versions already installed in the runtime. Python 3.10 or later is required. This is a Colab-oriented environment, not a complete dependency lockfile. Use a fresh runtime for this notebook; its dependency pins differ from the Transformers-based ESM-2 workflow. Restart after installation if Colab requests it.

The following cells initialize seed `44`, select CPU/CUDA, mount Google Drive, and define paths and hyperparameters. The default `esmc_600m` representation has 1,152 features; `esmc_300m` has 960. Consequently, the symmetric classifier input has 3,456 or 2,880 features, respectively. Changing the encoder changes the first layer's parameter count.

Reference implementations: [EvolutionaryScale ESM](https://github.com/evolutionaryscale/esm) and [ESM package version 3.2.1](https://pypi.org/project/esm/3.2.1/). Consult their model terms when using pretrained weights.


In [ ]:
import sys, subprocess, importlib.metadata as metadata
from pathlib import Path
if sys.version_info < (3, 10):
    raise RuntimeError("Use Python 3.10 or newer in Colab.")
constraints = []
for package in ("torch", "torchvision", "numpy", "pandas", "matplotlib", "scikit-learn"):
    try:
        constraints.append(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        pass
constraint_path = Path("/tmp/snooppi_esmc_constraints.txt")
constraint_path.write_text("\n".join(constraints))
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                      "-c", str(constraint_path), "esm==3.2.1", "transformers==4.48.1"])
print("Setup complete. Restart the session if Colab requests it.")


In [ ]:
import os
import random
import re
from contextlib import nullcontext
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from google.colab import drive
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from torch.utils.data import DataLoader, Dataset
import gc
import json
import hashlib
import warnings
import importlib.metadata as metadata
from IPython.display import display

SEED = 44

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

try:
    torch.set_float32_matmul_precision("high")
except AttributeError:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)


In [ ]:
# Mount Google Drive once.
if not Path("/content/drive/MyDrive").exists():
    drive.mount("/content/drive")

ROOT = Path("/content/drive/MyDrive/SNOOPPI")
SPLITS_DIR = ROOT / "splits"
EMBEDDINGS_DIR = ROOT / "embeddings"
CHECKPOINTS_DIR = ROOT / "checkpoints"
TABLES_DIR = ROOT / "tables"
FIGURES_DIR = ROOT / "figures"

for directory in [
    EMBEDDINGS_DIR,
    CHECKPOINTS_DIR,
    TABLES_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

SPLIT_PATHS = {
    "train": SPLITS_DIR / "train.csv",
    "validation": SPLITS_DIR / "validation.csv",
    "test": SPLITS_DIR / "test.csv",
}

MODEL_TAG = "esmc_600m"  # Alternative: "esmc_300m"; separate caches/checkpoints.
if MODEL_TAG not in {"esmc_600m", "esmc_300m"}:
    raise ValueError("MODEL_TAG must be esmc_600m or esmc_300m")
MODEL_ID = MODEL_TAG
EMBEDDING_DIM = {"esmc_600m": 1152, "esmc_300m": 960}[MODEL_TAG]
PAIR_FEATURE_DIM = 3 * EMBEDDING_DIM
MAX_EPOCHS = 50
PATIENCE = 10
RESUME_TRAINING = True
RUN_SUFFIX = ""  # Use e.g. "_seed45" for a separate classifier experiment.
MAX_RESIDUES = 1024
EMBEDDING_BATCH_SIZE = 1  # Conservative for T4; increase after checking memory.
CACHE_SAVE_EVERY = 1_000
CLASSIFIER_BATCH_SIZE = 1024

EMBEDDING_CACHE_PATH = (
    EMBEDDINGS_DIR / f"{MODEL_TAG}_snooppi_mean_pool_embeddings.pt"
)
BEST_CHECKPOINT_PATH = (
    CHECKPOINTS_DIR / f"{MODEL_TAG}{RUN_SUFFIX}_best_classifier.pt"
)

print("Training split:", SPLIT_PATHS["train"])
print("Embedding cache:", EMBEDDING_CACHE_PATH)
print("Best checkpoint:", BEST_CHECKPOINT_PATH)
LAST_CHECKPOINT_PATH = CHECKPOINTS_DIR / f"{MODEL_TAG}{RUN_SUFFIX}_last_classifier.pt"
RUN_TAG = MODEL_TAG + RUN_SUFFIX
print("GPU:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU (slow)")


## 2. Load saved splits and audit raw-sequence overlap

The loader retains row order and partition membership. It requires nonmissing, nonempty protein sequences and both binary classes in each split. If `label` is absent, it maps `positive → 1` and `negative → 0` from `SNOOPPI_final_label`; unmapped values fail the subsequent binary-label check. If a numeric `label` column already exists, the code uses it without checking its agreement with the original SNOOPPI label column.

The audit records SHA-256 hashes of the three input files and counts shared **raw** sequences and unordered raw-sequence pairs across splits. Pair overlap emits a warning but does not stop execution or remove rows. Shared proteins preclude interpreting the result as a cold-protein evaluation. This audit does not test sequence-similarity clusters, within-split duplicates, or collisions introduced by normalization/truncation; these require separate analysis before strong generalization claims.


In [ ]:
REQUIRED_COLUMNS = {
    "partner_A_sequence",
    "partner_B_sequence",
}


def load_split(path):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Keep the same saved split used by ProtT5; "
            "do not silently create a new split for this benchmark."
        )

    dataframe = pd.read_csv(path)
    missing_columns = REQUIRED_COLUMNS - set(dataframe.columns)
    if missing_columns:
        raise ValueError(
            f"{path.name} is missing columns: {sorted(missing_columns)}"
        )

    if "label" not in dataframe.columns:
        if "SNOOPPI_final_label" not in dataframe.columns:
            raise ValueError(
                f"{path.name} needs either 'label' or "
                "'SNOOPPI_final_label'."
            )
        dataframe["label"] = (
            dataframe["SNOOPPI_final_label"]
            .astype(str)
            .str.lower()
            .map({"negative": 0.0, "positive": 1.0})
        )

    if dataframe[list(REQUIRED_COLUMNS)].isna().any().any():
        raise ValueError(f"{path.name} contains missing protein sequences.")

    dataframe["label"] = pd.to_numeric(
        dataframe["label"], errors="raise"
    ).astype(np.float32)

    observed_labels = set(dataframe["label"].unique().tolist())
    if not observed_labels.issubset({0.0, 1.0}):
        raise ValueError(
            f"{path.name} has non-binary labels: {observed_labels}"
        )

    if dataframe.empty or dataframe["label"].nunique() != 2:
        raise ValueError(f"{path.name} must contain both binary classes for AUROC/AP.")
    if (dataframe[list(REQUIRED_COLUMNS)].astype(str).apply(lambda s: s.str.strip().eq("")).any().any()):
        raise ValueError(f"{path.name} contains empty sequences.")
    return dataframe.reset_index(drop=True)


train_df = load_split(SPLIT_PATHS["train"])
val_df = load_split(SPLIT_PATHS["validation"])
test_df = load_split(SPLIT_PATHS["test"])

for split_name, dataframe in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
]:
    counts = dataframe["label"].value_counts().sort_index().to_dict()
    print(f"{split_name:10s}: {len(dataframe):,} pairs | labels {counts}")


In [ ]:
split_frames = {"train": train_df, "validation": val_df, "test": test_df}
SPLIT_HASHES = {name: hashlib.sha256(path.read_bytes()).hexdigest()
                for name, path in SPLIT_PATHS.items()}
def protein_set(df):
    return set(df["partner_A_sequence"]) | set(df["partner_B_sequence"])
def pair_set(df):
    return {tuple(sorted((a, b))) for a, b in zip(df.partner_A_sequence, df.partner_B_sequence)}
for left, right in [("train", "validation"), ("train", "test"), ("validation", "test")]:
    a, b = split_frames[left], split_frames[right]
    n_proteins = len(protein_set(a) & protein_set(b))
    n_pairs = len(pair_set(a) & pair_set(b))
    print(f"{left}/{right}: shared raw sequences={n_proteins:,}; shared unordered pairs={n_pairs:,}")
    if n_pairs:
        warnings.warn("Pair overlap detected. Retained to reproduce the existing benchmark; disclose leakage.")


## 3. Normalize sequences and validate the embedding cache

`normalize_sequence()` uppercases each sequence, removes literal spaces, replaces every noncanonical character with `X`, and retains the first `MAX_RESIDUES = 1024` residues. The later pooling mask includes `X` residues but excludes BOS, EOS, and padding. Prefix truncation can remove interaction-relevant regions. Distinct raw sequences may also become identical model inputs after normalization or truncation.

Unique raw sequences from all three splits are sorted by normalized length and used as cache keys. Encoding validation/test sequences with a frozen pretrained model does not fit the model to their labels. The cache stores CPU FP16 vectors and metadata for the model, normalization rule, ESM version, sequence limit, dimension, and pooling description. Loading checks the model/normalization/version/limit/dimension fields and checks required vectors for shape and finiteness. The code writes through a temporary file before replacement.

The companion ProtT5 workflow uses a different normalization rule. Accordingly, this is a comparison of the supplied workflows; it is not an isolated test of encoder identity under fully identical preprocessing.


In [ ]:
def normalize_sequence(sequence):
    sequence = str(sequence).upper().replace(" ", "")
    # Match the existing ESM-2 preprocessing.
    sequence = re.sub(r"[^ACDEFGHIKLMNPQRSTVWY]", "X", sequence)
    return sequence[:MAX_RESIDUES]


all_sequences = pd.unique(
    pd.concat(
        [
            train_df["partner_A_sequence"],
            train_df["partner_B_sequence"],
            val_df["partner_A_sequence"],
            val_df["partner_B_sequence"],
            test_df["partner_A_sequence"],
            test_df["partner_B_sequence"],
        ],
        ignore_index=True,
    )
).tolist()

# Similar lengths in the same mini-batch reduce padding and GPU memory use.
all_sequences.sort(key=lambda sequence: len(normalize_sequence(sequence)))
print(f"Unique protein sequences: {len(all_sequences):,}")


In [ ]:
def atomic_torch_save(payload, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary_path = path.with_name(path.name + ".tmp")
    torch.save(payload, temporary_path)
    os.replace(temporary_path, path)


def load_trusted_torch_file(path, map_location="cpu"):
    # These files are created by this notebook, so full checkpoint loading
    # is intentional. The fallback keeps compatibility with older PyTorch.
    try:
        return torch.load(
            path,
            map_location=map_location,
            weights_only=False,
        )
    except TypeError:
        return torch.load(path, map_location=map_location)


def save_embedding_cache(embedding_cache):
    payload = {
        "model_id": MODEL_ID,
        "normalization": "canonical_X_spaces_removed_prefix_v1",
        "esm_version": "3.2.1",
        "max_residues": MAX_RESIDUES,
        "embedding_dim": EMBEDDING_DIM,
        "pooling": "mean over non-special residue tokens",
        "embeddings": embedding_cache,
    }
    atomic_torch_save(payload, EMBEDDING_CACHE_PATH)


def load_embedding_cache():
    if not EMBEDDING_CACHE_PATH.exists():
        return {}

    payload = load_trusted_torch_file(EMBEDDING_CACHE_PATH)
    expected_metadata = {
        "model_id": MODEL_ID,
        "normalization": "canonical_X_spaces_removed_prefix_v1",
        "esm_version": "3.2.1",
        "max_residues": MAX_RESIDUES,
        "embedding_dim": EMBEDDING_DIM,
    }

    for key, expected_value in expected_metadata.items():
        if payload.get(key) != expected_value:
            raise ValueError(
                f"Embedding cache mismatch for {key}: "
                f"found {payload.get(key)!r}, expected {expected_value!r}."
            )

    cache = payload["embeddings"]
    print(f"Loaded {len(cache):,} cached ESM-C embeddings.")
    return cache


In [ ]:
embedding_cache = load_embedding_cache()
for seq in all_sequences:
    if not normalize_sequence(seq):
        raise ValueError("A protein becomes empty after sequence normalization.")
    if seq in embedding_cache:
        value = embedding_cache[seq]
        if value.shape != (EMBEDDING_DIM,) or not torch.isfinite(value).all():
            raise ValueError("Invalid cached embedding: remove/rebuild the affected ESM-C cache.")
missing_sequences = [seq for seq in all_sequences if seq not in embedding_cache]
print(f"Cached: {len(all_sequences)-len(missing_sequences):,}; missing: {len(missing_sequences):,}")
print("Truncated proteins:", sum(len(str(s).replace(" ", "")) > MAX_RESIDUES for s in all_sequences))


## 4. Load the frozen ESM-C encoder when needed

The encoder is loaded only if required sequences are missing from the cache. The code uses the pinned package's model builders with Flash Attention disabled, initializes on CPU, and then transfers the model to the selected device. It chooses BF16 on CUDA hardware reporting native support, otherwise FP16 on CUDA; CPU execution uses FP32. Model parameters are frozen and evaluation mode is enabled.

Embedding inference calls the encoder's `forward` interface directly. The cache metadata does not distinguish inference precision or a specific pretrained-weight revision, so keep those choices consistent within an experiment and use a separate artifact path if intentionally changing them.

If model access fails with HTTP 401/403, verify access to the relevant Hugging Face model and authenticate with `huggingface_hub.notebook_login()` in a separate unsaved cell. Do not store access tokens in the notebook.


In [ ]:
encoder = None
if missing_sequences:
    from esm.pretrained import ESMC_300M_202412, ESMC_600M_202412
    builder = ESMC_600M_202412 if MODEL_TAG == "esmc_600m" else ESMC_300M_202412
    encoder = builder(device="cpu", use_flash_attn=False)
    encoder_dtype = (torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16) if DEVICE.type == "cuda" else torch.float32
    encoder = encoder.to(device=DEVICE, dtype=encoder_dtype).eval()
    for parameter in encoder.parameters():
        parameter.requires_grad_(False)
    tokenizer = encoder.tokenizer
    assert encoder.embed.embedding_dim == EMBEDDING_DIM
    print("Loaded:", MODEL_TAG, "dtype:", encoder_dtype)
else:
    print("Complete cache found; model loading skipped.")


## 5. Compute residue-mean embeddings and save progress

The preserved token mask explicitly selects residue positions `1 … sequence_length`. The token-count assertion checks the expected BOS/residue/EOS layout; pooling excludes BOS, EOS, and padding while retaining `X`. Each cached vector is cloned so it does not retain its batch's larger storage allocation.

**Current supplied setting:** `CACHE_SAVE_EVERY = 1_000`, meaning a save after at least 1,000 newly embedded proteins, at completion, and through the `finally` block when Python unwinds an exception or interrupt. This edition keeps that value unchanged. A hard runtime disconnect can lose work since the last successful save.

GPU out-of-memory handling retries with half the batch size. At batch size 1, it stops with an explanatory error and attempts to save the cache. Preserve normalization and sequence length when resuming; a change defines a different embedding experiment. Once all required vectors are available, the encoder is released before classifier training.


In [ ]:
@torch.inference_mode()
def embed_batch(original_batch):
    cleaned = [normalize_sequence(s) for s in original_batch]
    tokens = tokenizer(cleaned, add_special_tokens=True, padding=True, return_tensors="pt")
    ids = tokens["input_ids"].to(DEVICE)
    output = encoder(sequence_tokens=ids)
    hidden = output.embeddings.float()
    # Known residue spans exclude BOS, EOS, and any padding, while retaining X.
    positions = torch.arange(ids.shape[1], device=DEVICE)[None, :]
    lengths = torch.tensor([len(s) for s in cleaned], device=DEVICE)[:, None]
    expected_tokens = lengths.squeeze(1) + 2
    assert torch.equal((ids != tokenizer.pad_token_id).sum(1), expected_tokens)
    mask = (positions >= 1) & (positions <= lengths)
    pooled = (hidden * mask.unsqueeze(-1)).sum(1) / lengths
    if not torch.isfinite(pooled).all():
        raise FloatingPointError("Nonfinite embeddings. Retry in FP32 with a separate cache.")
    # clone avoids retaining the whole batch's storage for each cached vector.
    return [row.cpu().half().clone() for row in pooled]

if missing_sequences:
    cursor, unsaved = 0, 0
    batch_size = EMBEDDING_BATCH_SIZE
    try:
        while cursor < len(missing_sequences):
            batch = missing_sequences[cursor:cursor + batch_size]
            try:
                vectors = embed_batch(batch)
            except torch.cuda.OutOfMemoryError:
                gc.collect()
                torch.cuda.empty_cache()
                if batch_size == 1:
                    raise RuntimeError("GPU OOM at batch size 1. Free GPU memory or use a larger GPU / separate 300M run.") from None
                batch_size = max(1, batch_size // 2)
                print("Retrying batch size:", batch_size)
                continue
            embedding_cache.update(zip(batch, vectors))
            cursor += len(batch)
            unsaved += len(batch)
            if unsaved >= CACHE_SAVE_EVERY or cursor == len(missing_sequences):
                save_embedding_cache(embedding_cache)
                unsaved = 0
                print(f"Embedded {cursor:,}/{len(missing_sequences):,} new proteins; saved.")
    finally:
        save_embedding_cache(embedding_cache)
assert all(s in embedding_cache for s in all_sequences)
print("Complete cache:", len(all_sequences), "required proteins")
del encoder
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()


## 6. Construct symmetric pair features and the classifier

For protein vectors $a$ and $b$, the input is

$$x_{AB} = [a+b,\; |a-b|,\; a\odot b].$$

This representation is invariant to swapping the two partners. `PPIPairDataset` retrieves cached vectors, converts them to FP32, and constructs features on demand. Training batches are shuffled with a seeded generator; validation/test batches retain row order.

The classifier has hidden widths **512 → 512 → 256**, ReLU activations, dropout `0.1` after each hidden layer, and one logit output. AdamW uses learning rate `1e-4` and weight decay `1e-4`; classifier batch size is `1024`. Weighted binary cross-entropy uses `n_negative / n_positive` from the training labels only. Seeds are reset before classifier construction so encoder loading does not advance its initialization state.


In [ ]:
class PPIPairDataset(Dataset):
    def __init__(self, dataframe, embedding_cache):
        self.dataframe = dataframe.reset_index(drop=True)
        self.embedding_cache = embedding_cache

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        embedding_a = self.embedding_cache[
            row["partner_A_sequence"]
        ].float()
        embedding_b = self.embedding_cache[
            row["partner_B_sequence"]
        ].float()

        pair_features = torch.cat(
            [
                embedding_a + embedding_b,
                torch.abs(embedding_a - embedding_b),
                embedding_a * embedding_b,
            ]
        )
        label = torch.tensor(row["label"], dtype=torch.float32)
        return pair_features, label


train_dataset = PPIPairDataset(train_df, embedding_cache)
val_dataset = PPIPairDataset(val_df, embedding_cache)
test_dataset = PPIPairDataset(test_df, embedding_cache)

loader_settings = {
    "batch_size": CLASSIFIER_BATCH_SIZE,
    "num_workers": 0,
    "pin_memory": DEVICE.type == "cuda",
}

shuffle_generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(
    train_dataset,
    shuffle=True,
    generator=shuffle_generator,
    **loader_settings,
)
val_loader = DataLoader(val_dataset, shuffle=False, **loader_settings)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_settings)

features, labels = next(iter(val_loader))
print("Feature batch:", tuple(features.shape))
print("Label batch:", tuple(labels.shape))
assert features.shape[1] == PAIR_FEATURE_DIM


In [ ]:
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)
class PPIClassifier(nn.Module):
    def __init__(self, embedding_dim):
        super().__init__()

        self.embedding_dim = int(embedding_dim)
        input_dimension = 3 * self.embedding_dim

        self.network = nn.Sequential(
            nn.Linear(input_dimension, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.1),

            nn.Linear(256, 1),
        )

    def forward(self, features):
        return self.network(features).squeeze(-1)


model = PPIClassifier(EMBEDDING_DIM).to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-4,
)

n_positive = int(train_df["label"].sum())
n_negative = len(train_df) - n_positive
positive_weight = n_negative / n_positive

criterion = nn.BCEWithLogitsLoss(
    pos_weight=torch.tensor([positive_weight], device=DEVICE)
)

print(model)
print(f"Positive weight: {positive_weight:.6f}")


## 7. Train with validation selection and epoch-level recovery

Training runs for at most 50 epochs with patience 10. After each epoch, the code evaluates validation performance and saves a new best checkpoint only when **validation macro-AP** improves. Macro-AP is the mean of positive-class AP and negative-class AP computed from the complementary scores.

The last checkpoint records the most recently completed epoch, model and optimizer states, training history, early-stopping state, and Python/NumPy/PyTorch/shuffle-generator RNG states. Recovery requires a matching `RUN_CONFIG`, including hashes of the split files, and an existing best checkpoint. It resumes at the next epoch; a partially completed epoch is not recovered. Completed or early-stopped runs proceed to evaluation.

Use a new `RUN_SUFFIX` for an intentionally separate classifier experiment. The implementation remains a single-seed study; initialization control and checkpoint recovery alone do not establish identical results across hardware or software versions. Keep the designated test split out of model and hyperparameter selection.


In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    all_labels = []
    all_probabilities = []

    gradient_context = torch.enable_grad() if training else torch.no_grad()
    with gradient_context:
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            if training:
                optimizer.zero_grad(set_to_none=True)

            logits = model(features)
            loss = criterion(logits, labels)

            if training:
                loss.backward()
                optimizer.step()

            probabilities = torch.sigmoid(logits)
            total_loss += loss.item() * labels.size(0)
            all_labels.append(labels.detach().cpu().numpy())
            all_probabilities.append(
                probabilities.detach().cpu().numpy()
            )

    labels = np.concatenate(all_labels).astype(int)
    probabilities = np.concatenate(all_probabilities)
    positive_ap = average_precision_score(labels, probabilities)
    negative_ap = average_precision_score(1 - labels, 1 - probabilities)

    return {
        "loss": total_loss / len(loader.dataset),
        "auroc": roc_auc_score(labels, probabilities),
        "positive_ap": positive_ap,
        "negative_ap": negative_ap,
        "macro_ap": (positive_ap + negative_ap) / 2,
    }


In [ ]:
RUN_CONFIG = {"model_id": MODEL_ID, "embedding_dim": EMBEDDING_DIM,
              "max_residues": MAX_RESIDUES, "seed": SEED,
              "split_sha256": SPLIT_HASHES, "batch_size": CLASSIFIER_BATCH_SIZE,
              "normalization": "canonical_X_spaces_removed_prefix_v1", "esm_version": "3.2.1",
              "lr": 1e-4, "weight_decay": 1e-4, "patience": PATIENCE,
              "pair_features": ["sum", "absolute_difference", "product"],
              "selection_metric": "validation macro-AP"}
history, start_epoch = [], 1
best_val_macro_ap, epochs_without_improvement = -float("inf"), 0
if LAST_CHECKPOINT_PATH.exists() and RESUME_TRAINING:
    last = load_trusted_torch_file(LAST_CHECKPOINT_PATH)
    if last["run_config"] != RUN_CONFIG:
        raise ValueError("Run configuration changed. Restore it or use a new RUN_SUFFIX.")
    if not BEST_CHECKPOINT_PATH.exists():
        raise FileNotFoundError("Last checkpoint exists without best checkpoint. Restore best or use a new run suffix.")
    model.load_state_dict(last["model_state_dict"])
    optimizer.load_state_dict(last["optimizer_state_dict"])
    history = last["history"]
    start_epoch = last["epoch"] + 1
    best_val_macro_ap = last["best_val_macro_ap"]
    epochs_without_improvement = last["epochs_without_improvement"]
    random.setstate(last["python_rng"])
    np.random.set_state(last["numpy_rng"])
    torch.set_rng_state(last["torch_rng"])
    shuffle_generator.set_state(last["shuffle_rng"])
    if DEVICE.type == "cuda" and last["cuda_rng"] is not None:
        torch.cuda.set_rng_state_all(last["cuda_rng"])
    print("Resuming at epoch", start_epoch)
    del last
elif BEST_CHECKPOINT_PATH.exists() or LAST_CHECKPOINT_PATH.exists():
    raise FileExistsError("Existing run found. Enable RESUME_TRAINING or choose a new RUN_SUFFIX.")

for epoch in range(start_epoch, MAX_EPOCHS + 1):
    if epochs_without_improvement >= PATIENCE:
        print("Early stopping was already reached; reusing the best checkpoint.")
        break
    train_metrics = run_epoch(model, train_loader, criterion, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, criterion)
    if not np.isfinite(val_metrics["macro_ap"]):
        raise FloatingPointError("Validation macro-AP is not finite.")
    history.append({"epoch": epoch, "train_loss": train_metrics["loss"],
                    "train_auroc": train_metrics["auroc"], "train_macro_ap": train_metrics["macro_ap"],
                    "val_loss": val_metrics["loss"], "val_auroc": val_metrics["auroc"],
                    "val_macro_ap": val_metrics["macro_ap"]})
    improved = val_metrics["macro_ap"] > best_val_macro_ap
    if improved:
        best_val_macro_ap = val_metrics["macro_ap"]
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
    checkpoint = {"epoch": epoch, "run_config": RUN_CONFIG,
                  "model_id": MODEL_ID, "embedding_dim": EMBEDDING_DIM,
                  "pair_feature_dim": PAIR_FEATURE_DIM, "max_residues": MAX_RESIDUES,
                  "model_state_dict": model.state_dict(), "optimizer_state_dict": optimizer.state_dict(),
                  "best_val_macro_ap": float(best_val_macro_ap), "validation_metrics": val_metrics,
                  "history": history, "epochs_without_improvement": epochs_without_improvement,
                  "python_rng": random.getstate(), "numpy_rng": np.random.get_state(),
                  "torch_rng": torch.get_rng_state(), "shuffle_rng": shuffle_generator.get_state(),
                  "cuda_rng": torch.cuda.get_rng_state_all() if DEVICE.type == "cuda" else None}
    if improved:
        atomic_torch_save(checkpoint, BEST_CHECKPOINT_PATH)
    atomic_torch_save(checkpoint, LAST_CHECKPOINT_PATH)
    pd.DataFrame(history).to_csv(TABLES_DIR / f"{RUN_TAG}_training_history.csv", index=False)
    print(f"Epoch {epoch:02d} | train loss {train_metrics['loss']:.4f} | val loss {val_metrics['loss']:.4f} | val AUROC {val_metrics['auroc']:.4f} | val macro-AP {val_metrics['macro_ap']:.4f}" + (" | saved best" if improved else ""))

history_df = pd.DataFrame(history)
assert BEST_CHECKPOINT_PATH.exists(), "No best checkpoint available. Run training first."
print("Best validation macro-AP:", round(best_val_macro_ap, 4))


## 8. Select the threshold on validation and evaluate the chosen model

The best validation-macro-AP checkpoint is reloaded. The decision threshold is selected by maximizing **validation macro-F1** over 181 equally spaced values from `0.05` through `0.95`; it is then applied unchanged to all splits. The test split does not select the checkpoint or threshold.

Saved metrics include weighted loss, AUROC, positive AP, negative AP, macro-AP, macro-F1, Matthews correlation coefficient, balanced accuracy, positive-class precision, and positive-class recall. Report class counts/prevalence alongside AP when interpreting class imbalance. AP here is scikit-learn's average precision and should not be silently substituted for a differently computed area under a precision–recall curve.

The cell exports metrics, test scores/predictions, and a confusion matrix; it also adds the selected threshold and split metrics to the best checkpoint and writes a weights-only file. Test `row_index` refers to row order in the saved test CSV, whose hash is recorded in the configuration. Sigmoid outputs from weighted BCE are **interaction scores**, not calibrated binding probabilities or binding affinities.


In [ ]:
best_checkpoint = load_trusted_torch_file(
    BEST_CHECKPOINT_PATH,
    map_location=DEVICE,
)
model.load_state_dict(best_checkpoint["model_state_dict"])
model.eval()


def collect_predictions(model, loader, criterion):
    labels_all = []
    probabilities_all = []
    total_loss = 0.0

    with torch.no_grad():
        for features, labels in loader:
            features = features.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            logits = model(features)
            loss = criterion(logits, labels)
            probabilities = torch.sigmoid(logits)

            total_loss += loss.item() * labels.size(0)
            labels_all.append(labels.cpu().numpy())
            probabilities_all.append(probabilities.cpu().numpy())

    return (
        total_loss / len(loader.dataset),
        np.concatenate(labels_all).astype(int),
        np.concatenate(probabilities_all),
    )


def summarize_predictions(loss, labels, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    positive_ap = average_precision_score(labels, probabilities)
    negative_ap = average_precision_score(1 - labels, 1 - probabilities)

    return {
        "loss": loss,
        "auroc": roc_auc_score(labels, probabilities),
        "positive_ap": positive_ap,
        "negative_ap": negative_ap,
        "macro_ap": (positive_ap + negative_ap) / 2,
        "macro_f1": f1_score(labels, predictions, average="macro"),
        "mcc": matthews_corrcoef(labels, predictions),
        "balanced_accuracy": balanced_accuracy_score(labels, predictions),
        "precision": precision_score(
            labels, predictions, zero_division=0
        ),
        "recall": recall_score(labels, predictions, zero_division=0),
    }


train_loss, train_labels, train_probabilities = collect_predictions(
    model, DataLoader(train_dataset, shuffle=False, **loader_settings), criterion
)
val_loss, val_labels, val_probabilities = collect_predictions(
    model, val_loader, criterion
)

# Select the classification threshold using validation data only.
threshold_grid = np.linspace(0.05, 0.95, 181)
decision_threshold = max(
    threshold_grid,
    key=lambda threshold: f1_score(
        val_labels,
        val_probabilities >= threshold,
        average="macro",
    ),
)

test_loss, test_labels, test_probabilities = collect_predictions(
    model, test_loader, criterion
)

split_results = {
    "Train": summarize_predictions(
        train_loss,
        train_labels,
        train_probabilities,
        decision_threshold,
    ),
    "Validation": summarize_predictions(
        val_loss,
        val_labels,
        val_probabilities,
        decision_threshold,
    ),
    "Test": summarize_predictions(
        test_loss,
        test_labels,
        test_probabilities,
        decision_threshold,
    ),
}

split_metrics = (
    pd.DataFrame(split_results)
    .T
    .rename_axis("split")
    .reset_index()
)
split_metrics.to_csv(
    TABLES_DIR / f"{RUN_TAG}_split_metrics.csv",
    index=False,
)

test_predictions = pd.DataFrame(
    {
        "row_index": np.arange(len(test_labels)),
        "label": test_labels,
        "interaction_score": test_probabilities,
        "prediction": (
            test_probabilities >= decision_threshold
        ).astype(int),
    }
)
test_predictions.to_csv(
    TABLES_DIR / f"{RUN_TAG}_test_predictions.csv",
    index=False,
)

test_confusion_matrix = confusion_matrix(
    test_labels,
    test_predictions["prediction"],
    labels=[0, 1],
)
pd.DataFrame(
    test_confusion_matrix,
    index=["true_negative", "true_positive"],
    columns=["predicted_negative", "predicted_positive"],
).to_csv(TABLES_DIR / f"{RUN_TAG}_test_confusion_matrix.csv")

# Add the validation-selected threshold and final metrics to the saved
# best checkpoint, then write a separate weights-only file.
best_checkpoint["decision_threshold"] = float(decision_threshold)
best_checkpoint["split_metrics"] = split_metrics.set_index(
    "split"
).to_dict("index")
atomic_torch_save(best_checkpoint, BEST_CHECKPOINT_PATH)
atomic_torch_save(
    model.state_dict(),
    CHECKPOINTS_DIR / f"{RUN_TAG}_best_classifier_weights.pt",
)

display(split_metrics.round(4))
print(f"Validation-selected threshold: {decision_threshold:.3f}")
print(f"Selected epoch: {best_checkpoint['epoch']}")


## 9. Export training and evaluation figures

The figure cell writes 300-dpi PNGs for training/validation loss, validation AUROC/macro-AP, and the test confusion matrix. Dashed epoch markers come from the saved best checkpoint. PNG export preserves the supplied workaround for PDF-backend problems.

Training-history loss is aggregated during optimization, whereas validation loss is evaluated after the epoch; these curves therefore use different model states. Final split metrics in section 8 are computed with the selected checkpoint in evaluation mode.


In [ ]:
# Section 9: Save training and evaluation figures

import matplotlib.pyplot as plt

plt.rcParams.update(
    {
        "font.family": "serif",
        "font.size": 11,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "figure.dpi": 150,
    }
)

selected_epoch = int(best_checkpoint["epoch"])


# ============================================================
# Figure 1: Training curves and validation performance
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.3))

# Training and validation loss
axes[0].plot(
    history_df["epoch"],
    history_df["train_loss"],
    linewidth=2,
    label="Training",
)

axes[0].plot(
    history_df["epoch"],
    history_df["val_loss"],
    linewidth=2,
    label="Validation",
)

axes[0].axvline(
    selected_epoch,
    color="black",
    linestyle="--",
    linewidth=1,
    label=f"Selected epoch: {selected_epoch}",
)

axes[0].set(
    title="ESM-C training and validation loss",
    xlabel="Epoch",
    ylabel="Weighted binary cross-entropy",
)

axes[0].grid(alpha=0.2)
axes[0].legend(frameon=False)


# Validation AUROC and macro-AP
axes[1].plot(
    history_df["epoch"],
    history_df["val_auroc"],
    linewidth=2,
    label="AUROC",
)

axes[1].plot(
    history_df["epoch"],
    history_df["val_macro_ap"],
    linewidth=2,
    label="Macro-AP",
)

axes[1].axvline(
    selected_epoch,
    color="black",
    linestyle="--",
    linewidth=1,
)

axes[1].set(
    title="ESM-C validation performance",
    xlabel="Epoch",
    ylabel="Score",
    ylim=(0.0, 1.0),
)

axes[1].grid(alpha=0.2)
axes[1].legend(frameon=False)

fig.tight_layout()

training_curve_path = (
    FIGURES_DIR / f"{RUN_TAG}_training_curves.png"
)

fig.savefig(
    training_curve_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved training curves to: {training_curve_path}")


# ============================================================
# Figure 2: Test confusion matrix
# ============================================================

fig, ax = plt.subplots(figsize=(5.2, 4.5))

image = ax.imshow(
    test_confusion_matrix,
    cmap="Blues",
)

maximum_count = test_confusion_matrix.max()

for row in range(2):
    for column in range(2):
        value = test_confusion_matrix[row, column]

        ax.text(
            column,
            row,
            f"{value:,}",
            ha="center",
            va="center",
            fontsize=12,
            color=(
                "white"
                if value > maximum_count / 2
                else "black"
            ),
        )

ax.set(
    xticks=[0, 1],
    yticks=[0, 1],
    xticklabels=[
        "Non-interaction",
        "Interaction",
    ],
    yticklabels=[
        "Non-interaction",
        "Interaction",
    ],
    xlabel="Predicted class",
    ylabel="True class",
    title=(
        "ESM-C test confusion matrix\n"
        f"Threshold = {decision_threshold:.2f}"
    ),
)

fig.colorbar(
    image,
    ax=ax,
    fraction=0.046,
    pad=0.04,
)

fig.tight_layout()

confusion_matrix_path = (
    FIGURES_DIR / f"{RUN_TAG}_test_confusion_matrix.png"
)

fig.savefig(
    confusion_matrix_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()
plt.close(fig)

print(f"Saved confusion matrix to: {confusion_matrix_path}")


## 10. Compare available result tables and save a run manifest

The code reads the legacy ProtT5 table `split_metrics.csv`, the ESM-2 table `esm2_t33_650M_split_metrics.csv`, and the current ESM-C result table. Missing files, missing split columns, and ambiguous test rows are skipped; missing metric columns remain missing. Scores are never filled from previous conversations or assumed results.

This comparison is **descriptive** until equivalent input rows, preprocessing, training histories, and metric definitions are established. The comparison cell does not verify hashes for legacy results, and macro-AP is not interchangeable with positive-class AP. The optional comparison figure is created only when more than one valid result row is available.

The run manifest records the configuration, maximum/selected epochs, validation threshold, Python version, selected package versions, and GPU name. It is written at this stage, after training and evaluation, rather than at run startup.


In [ ]:
comparison_paths = {"ProtT5": TABLES_DIR / "split_metrics.csv",
                    "ESM-2 650M": TABLES_DIR / "esm2_t33_650M_split_metrics.csv",
                    MODEL_TAG: TABLES_DIR / f"{RUN_TAG}_split_metrics.csv"}
comparison_rows = []
for name, path in comparison_paths.items():
    if not path.exists():
        print("Comparison skipped (missing):", path.name)
        continue
    table = pd.read_csv(path)
    if "split" not in table:
        print("Comparison skipped (missing split column):", path.name)
        continue
    rows = table.loc[table["split"].astype(str).str.lower().eq("test")]
    if len(rows) != 1:
        print("Comparison skipped (expected one Test row):", path.name)
        continue
    row = rows.iloc[0]
    comparison_rows.append({"model": name, **{metric: row.get(metric, np.nan)
                           for metric in ["auroc", "macro_ap", "macro_f1"]}})
comparison = pd.DataFrame(comparison_rows)
display(comparison.round(4))
comparison.to_csv(TABLES_DIR / f"{RUN_TAG}_benchmark_comparison.csv", index=False)
if len(comparison) > 1:
    ax = comparison.set_index("model")[["auroc", "macro_ap", "macro_f1"]].plot.bar(
        figsize=(9, 4.5), rot=0, color=["#176B87", "#D58632", "#6B5B95"])
    ax.set(ylim=(0, 1), ylabel="Test score", title="Frozen protein representations · descriptive comparison")
    ax.legend(["AUROC", "Macro-AP", "Macro-F1"], frameon=False)
    ax.figure.tight_layout()
    ax.figure.savefig(FIGURES_DIR / f"{RUN_TAG}_benchmark_comparison.png", dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(ax.figure)
manifest = {"run_config": RUN_CONFIG, "max_epochs": MAX_EPOCHS,
            "best_epoch": int(best_checkpoint["epoch"]), "decision_threshold": float(decision_threshold),
            "python": __import__("sys").version,
            "packages": {p: metadata.version(p) for p in ["esm", "torch", "transformers", "pandas", "numpy", "scikit-learn"]},
            "gpu": torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU"}
(TABLES_DIR / f"{RUN_TAG}_run_manifest.json").write_text(json.dumps(manifest, indent=2))


## 11. Check expected artifacts

The final code cell prints existence checks for the embedding cache, best/last checkpoints, weights-only checkpoint, training history, split metrics, test predictions/confusion matrix, run manifest, comparison table, and the two primary figures. It raises an error only if the best checkpoint is missing; the printed checks are not checksums or a full integrity test. The comparison figure is conditional and is not part of this fixed list.


In [ ]:
expected_outputs = [
    LAST_CHECKPOINT_PATH,
    TABLES_DIR / f"{RUN_TAG}_run_manifest.json",
    TABLES_DIR / f"{RUN_TAG}_benchmark_comparison.csv",
    EMBEDDING_CACHE_PATH,
    BEST_CHECKPOINT_PATH,
    CHECKPOINTS_DIR / f"{RUN_TAG}_best_classifier_weights.pt",
    TABLES_DIR / f"{RUN_TAG}_training_history.csv",
    TABLES_DIR / f"{RUN_TAG}_split_metrics.csv",
    TABLES_DIR / f"{RUN_TAG}_test_predictions.csv",
    TABLES_DIR / f"{RUN_TAG}_test_confusion_matrix.csv",
    FIGURES_DIR / f"{RUN_TAG}_training_curves.png",
    FIGURES_DIR / f"{RUN_TAG}_test_confusion_matrix.png",
]

for output_path in expected_outputs:
    print(f"{output_path.exists()!s:5s} | {output_path}")

if not BEST_CHECKPOINT_PATH.exists():
    raise RuntimeError("The best checkpoint was not saved successfully.")


### Recovery notes and interpretation limits

- **Embedding interruption:** rerun from the top with the same model, normalization, length limit, and paths. Only missing cache entries are recomputed. The current save interval is 1,000 new proteins.
- **Training interruption:** the last completed epoch is restored automatically when `RESUME_TRAINING = True`; the unfinished epoch is repeated.
- **CUDA memory:** the default embedding batch size is 1. Clear unrelated GPU allocations or use a separate 300M experiment if necessary. Keep 300M and 600M artifacts separate.
- **Imports:** start with a fresh Colab runtime and run the pinned setup. Avoid mixing this pinned environment with the unpinned dependency setup of the other notebooks.
- **Missing splits or mismatch errors:** restore the original files/configuration or select explicitly separate experiment paths. Do not bypass cache or checkpoint checks.
- **Unexpected performance:** examine label provenance, duplicates, normalized-sequence overlap, class balance, and preprocessing before attributing a difference to encoder quality.
- **Unresolved pairs:** any later scores are predictions, not verified interactions. Unknown labels must not be interpreted as experimentally established negatives.

The companion ProtT5 notebook retains multiple training attempts. Matching selected hyperparameters does not establish identical training histories. Repeated runs under a controlled protocol, and a clearly specified protein/cluster-disjoint evaluation when needed, would support stronger comparative claims. No claim of a completed ESM-C benchmark is made by this cleaned notebook.
